In [ ]:
import segmentation_models_pytorch
import albumentations

In [ ]:
source_folder_path = ''
destination_path = "/content/"
!cp -r "{source_folder_path}" "{destination_path}"
print(f"Folder copied from {source_folder_path} to {destination_path}")

In [ ]:
import torch
import os
class Config:
    ROOT_DIR = '/content/REFUGE'
    TRAIN_IMG_DIR = os.path.join(ROOT_DIR, 'train', 'images')
    TRAIN_MASK_DIR = os.path.join(ROOT_DIR, 'train', 'mask')
    VAL_IMG_DIR = os.path.join(ROOT_DIR, 'val', 'images')
    VAL_MASK_DIR = os.path.join(ROOT_DIR, 'val', 'mask')
    TEST_IMG_DIR = os.path.join(ROOT_DIR, 'test', 'images')
    TEST_MASK_DIR = os.path.join(ROOT_DIR, 'test', 'mask')
    ENCODER = 'efficientnet-b4'
    ENCODER_WEIGHTS = 'imagenet'
    CLASSES = ['background', 'disc', 'cup']
    ACTIVATION = None
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    INPUT_SIZE = None
    BATCH_SIZE = None
    LR = None
    EPOCHS = None

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
def get_training_augmentation():
    train_transform = [
        A.Resize(Config.INPUT_SIZE, Config.INPUT_SIZE, interpolation=cv2.INTER_AREA),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.Compose([
            A.RandomBrightnessContrast(brightness_limit=0.05, contrast_limit=0.05, p=0.5),
            A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=5, val_shift_limit=5, p=0.3),
        ]),
       
       A.HorizontalFlip(p=1),
       A.VerticalFlip(p=1),
       A.ElasticTransform(alpha=320, sigma=10, alpha_affine=50, p=1),
       A.GridDistortion(num_steps=5, distort_limit=0.3, p=1),
       A.Transpose(p=1),
       A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
    return A.Compose(train_transform)
def get_validation_augmentation():
    test_transform = [
        A.Resize(Config.INPUT_SIZE, Config.INPUT_SIZE, interpolation=cv2.INTER_AREA),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
    return A.Compose(test_transform)
print("Ultra-Safe Augmentations Loaded (No distortions).")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import torch
import os
class RefugeBaselineDataset(Dataset):
    def __init__(self, images_dir, masks_dir, augmentation=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        if not os.path.exists(images_dir) or not os.path.exists(masks_dir):
            raise FileNotFoundError(f"Directory not found. Check Config paths!\nImg: {images_dir}\nMsk: {masks_dir}")
        self.ids = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        self.augmentation = augmentation
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        img_name = self.ids[i]
        base_name = os.path.splitext(img_name)[0]
        mask_name = None
        for ext in ['.bmp', '.png', '.jpg', '.jpeg']:
            potential_path = os.path.join(self.masks_dir, base_name + ext)
            if os.path.exists(potential_path):
                mask_name = base_name + ext
                break
        if mask_name is None:
            mask_name = img_name
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)
        image = np.array(Image.open(img_path).convert("RGB"))
        try:
            mask_raw = np.array(Image.open(mask_path).convert("L"))
        except FileNotFoundError:
            raise FileNotFoundError(f"CRITICAL: Could not find mask for {img_name}. Checked: {mask_path}")
        mask_classes = np.zeros_like(mask_raw, dtype=np.uint8)
        mask_classes[mask_raw < 50] = 2
        mask_classes[(mask_raw >= 50) & (mask_raw <= 200)] = 1
        if self.augmentation:
            augmented = self.augmentation(image=image, mask=mask_classes)
            image = augmented['image']
            mask_classes = augmented['mask']
        if not isinstance(mask_classes, torch.Tensor):
            mask_classes = torch.from_numpy(mask_classes)
        mask_classes = mask_classes.long()
        return image, mask_classes
train_dataset = RefugeBaselineDataset(Config.TRAIN_IMG_DIR, Config.TRAIN_MASK_DIR, augmentation=get_training_augmentation())
val_dataset = RefugeBaselineDataset(Config.VAL_IMG_DIR, Config.VAL_MASK_DIR, augmentation=get_validation_augmentation())
test_dataset = RefugeBaselineDataset(Config.TEST_IMG_DIR, Config.TEST_MASK_DIR, augmentation=get_validation_augmentation())
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)
print(f"✅ Baseline Data Ready!")
print(f"   Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
import torch
import torch.nn.functional as F
import torch.cuda.amp as amp
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import segmentation_models_pytorch as smp
import os
import json
import time
from tqdm import tqdm
from google.colab import drive
DRIVE_SAVE_DIR = ''
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Save Directory Ready: {DRIVE_SAVE_DIR}")
def calculate_metrics(pred_mask, true_mask):
    iou_disc, iou_cup = 0.0, 0.0
    dice_disc, dice_cup = 0.0, 0.0
    prec_disc, prec_cup = 0.0, 0.0
    rec_disc, rec_cup = 0.0, 0.0
    batch_size = true_mask.size(0)
    smooth = 1e-6
    for i in range(batch_size):
        p = pred_mask[i]
        t = true_mask[i]
        p_c = (p == 2).float()
        t_c = (t == 2).float()
        tp_c = (p_c * t_c).sum()
        fp_c = (p_c * (1-t_c)).sum()
        fn_c = ((1-p_c) * t_c).sum()
        union_c = p_c.sum() + t_c.sum() - tp_c
        iou_cup += (tp_c + smooth) / (union_c + smooth)
        dice_cup += (2. * tp_c + smooth) / (p_c.sum() + t_c.sum() + smooth)
        prec_cup += (tp_c + smooth) / (tp_c + fp_c + smooth)
        rec_cup  += (tp_c + smooth) / (tp_c + fn_c + smooth)
        p_d = ((p == 1) | (p == 2)).float()
        t_d = ((t == 1) | (t == 2)).float()
        tp_d = (p_d * t_d).sum()
        fp_d = (p_d * (1-t_d)).sum()
        fn_d = ((1-p_d) * t_d).sum()
        union_d = p_d.sum() + t_d.sum() - tp_d
        iou_disc += (tp_d + smooth) / (union_d + smooth)
        dice_disc += (2. * tp_d + smooth) / (p_d.sum() + t_d.sum() + smooth)
        prec_disc += (tp_d + smooth) / (tp_d + fp_d + smooth)
        rec_disc  += (tp_d + smooth) / (tp_d + fn_d + smooth)
    return (iou_disc/batch_size, iou_cup/batch_size,
            dice_disc/batch_size, dice_cup/batch_size,
            prec_disc/batch_size, prec_cup/batch_size,
            rec_disc/batch_size, rec_cup/batch_size)
def train_one_epoch(model, loader, optimizer, loss_fn, scaler, scheduler):
    model.train()
    running_loss = 0.0
    acc_iou_d, acc_iou_c = 0.0, 0.0
    acc_prec_d, acc_prec_c = 0.0, 0.0
    acc_rec_d, acc_rec_c = 0.0, 0.0
    optimizer.zero_grad()
    loop = tqdm(loader, desc="Training", leave=False)
    for batch_idx, (images, masks) in enumerate(loop):
        images = images.to(Config.DEVICE)
        masks = masks.long().to(Config.DEVICE)
        with amp.autocast():
            outputs = model(images)
            loss = loss_fn(outputs, masks) / Config.ACCUMULATION_STEPS
        scaler.scale(loss).backward()
        if (batch_idx + 1) % Config.ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        running_loss += loss.item() * Config.ACCUMULATION_STEPS
        preds = torch.argmax(torch.softmax(outputs, dim=1), dim=1)
        iou_d, iou_c, _, _, prec_d, prec_c, rec_d, rec_c = calculate_metrics(preds, masks)
        acc_iou_d += iou_d.item()
        acc_iou_c += iou_c.item()
        acc_prec_d += prec_d.item()
        acc_prec_c += prec_c.item()
        acc_rec_d += rec_d.item()
        acc_rec_c += rec_c.item()
        current_lr = optimizer.param_groups[0]['lr']
        loop.set_postfix(loss=loss.item() * Config.ACCUMULATION_STEPS, lr=f"{current_lr:.2e}")
    length = len(loader)
    return (running_loss/length, acc_iou_d/length, acc_iou_c/length,
            acc_prec_d/length, acc_prec_c/length, acc_rec_d/length, acc_rec_c/length)
def validate_one_epoch(model, loader, loss_fn):
    model.eval()
    running_loss = 0.0
    acc_iou_d, acc_iou_c = 0.0, 0.0
    acc_dice_d, acc_dice_c = 0.0, 0.0
    acc_prec_d, acc_prec_c = 0.0, 0.0
    acc_rec_d, acc_rec_c = 0.0, 0.0
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="❄️ Validation", leave=False):
            images, masks = images.to(Config.DEVICE), masks.long().to(Config.DEVICE)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            running_loss += loss.item()
            preds = torch.argmax(torch.softmax(outputs, dim=1), dim=1)
            iou_d, iou_c, dice_d, dice_c, prec_d, prec_c, rec_d, rec_c = calculate_metrics(preds, masks)
            acc_iou_d += iu_d.item()
            acc_iou_c += iou_c.item()
            acc_dice_d += dice_d.item()
            acc_dice_c += dice_c.item()
            acc_prec_d += prec_d.item()
            acc_prec_c += prec_c.item()
            acc_rec_d += rec_d.item()
            acc_rec_c += rec_c.item()
    length = len(loader)
    return (running_loss/length, acc_iou_d/length, acc_iou_c/length,
            acc_dice_d/length, acc_dice_c/length,
            acc_prec_d/length, acc_prec_c/length,
            acc_rec_d/length, acc_rec_c/length)
print(f"Creating Model: {Config.ENCODER}...")
model = smp.UnetPlusPlus(
    encoder_name=Config.ENCODER, encoder_weights=Config.ENCODER_WEIGHTS,
    in_channels=3, classes=3, activation=None
).to(Config.DEVICE)
dice_loss = smp.losses.DiceLoss(mode='multiclass')
ce_loss = torch.nn.CrossEntropyLoss()
def criterion(pred, target): return 0.5 * dice_loss(pred, target) + 0.5 * ce_loss(pred, target)
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)
scaler = amp.GradScaler()
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-6)
history = []
log_file = os.path.join(DRIVE_SAVE_DIR, "training_log.txt")
json_file = os.path.join(DRIVE_SAVE_DIR, "metrics.json")
best_model_path = os.path.join(DRIVE_SAVE_DIR, "best_model.pth")
last_model_path = os.path.join(DRIVE_SAVE_DIR, "last_model.pth")
print(f"Starting Training for {Config.EPOCHS} Epochs...")
best_score = 0.0
for epoch in range(Config.EPOCHS):
    start_time = time.time()
    t_loss, t_iou_d, t_iou_c, t_prec_d, t_prec_c, t_rec_d, t_rec_c = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, scheduler)
    v_loss, v_iou_d, v_iou_c, v_dice_d, v_dice_c, v_prec_d, v_prec_c, v_rec_d, v_rec_c = validate_one_epoch(
        model, val_loader, criterion)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    avg_iou = (v_iou_d + v_iou_c) / 2
    duration = time.time() - start_time
    print(f"\nEpoch {epoch+1}/{Config.EPOCHS} | LR: {current_lr:.2e}")
    print(f"   Train Loss: {t_loss:.4f} | IoU(D/C): {t_iou_d:.2f}/{t_iou_c:.2f}")
    print(f"   Valid Loss: {v_loss:.4f} | IoU(D/C): {v_iou_d:.2f}/{v_iou_c:.2f} | Dice(D/C): {v_dice_d:.2f}/{v_dice_c:.2f}")
    print(f"   Valid Prec(D/C): {v_prec_d:.2f}/{v_prec_c:.2f} | Rec(D/C): {v_rec_d:.2f}/{v_rec_c:.2f}")
    epoch_metrics = {
        "epoch": epoch + 1,
        "lr": current_lr,
        "train_loss": t_loss,
        "train_iou_disc": t_iou_d, "train_iou_cup": t_iou_c,
        "train_prec_disc": t_prec_d, "train_prec_cup": t_prec_c,
        "train_rec_disc": t_rec_d, "train_rec_cup": t_rec_c,
        "val_loss": v_loss,
        "val_iou_disc": v_iou_d, "val_iou_cup": v_iou_c,
        "val_dice_disc": v_dice_d, "val_dice_cup": v_dice_c,
        "val_prec_disc": v_prec_d, "val_prec_cup": v_prec_c,
        "val_rec_disc": v_rec_d, "val_rec_cup": v_rec_c,
        "avg_iou": avg_iou
    }
    history.append(epoch_metrics)
    with open(json_file, 'w') as f:
        json.dump(history, f, indent=4)
    with open(log_file, "a") as f:
        f.write(f"E{epoch+1} | Loss: {v_loss:.4f} | IoU: {avg_iou:.4f} | Prec_D: {v_prec_d:.3f} | Rec_D: {v_rec_d:.3f}\n")
    torch.save(model.state_dict(), last_model_path)
    if avg_iou > best_score:
        best_score = avg_iou
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved New Best Model (IoU: {best_score:.4f})")

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os
import numpy as np
DRIVE_SAVE_DIR = ''
json_file = os.path.join(DRIVE_SAVE_DIR, "metrics.json")
def plot_training_history():
    if not os.path.exists(json_file):
        print(f"Log file not found at: {json_file}")
        return
    with open(json_file, 'r') as f:
        history = json.load(f)
    epochs = [x['epoch'] for x in history]
    lrs = [x['lr'] for x in history]
    train_loss = [x['train_loss'] for x in history]
    val_loss = [x['val_loss'] for x in history]
    val_iou_d = [x.get('val_iou_disc', 0) for x in history]
    val_iou_c = [x.get('val_iou_cup', 0) for x in history]
    val_dice_d = [x.get('val_dice_disc', 0) for x in history]
    val_dice_c = [x.get('val_dice_cup', 0) for x in history]
    val_prec_d = [x.get('val_prec_disc', 0) for x in history]
    val_prec_c = [x.get('val_prec_cup', 0) for x in history]
    val_rec_d = [x.get('val_rec_disc', 0) for x in history]
    val_rec_c = [x.get('val_rec_cup', 0) for x in history]
    fig, axs = plt.subplots(3, 2, figsize=(16, 18))
    plt.subplots_adjust(hspace=0.3, wspace=0.2)
    def make_smart_axis(ax, y_step=0.05, is_log=False):
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        if not is_log:
            ax.yaxis.set_major_locator(ticker.MultipleLocator(y_step))
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
            ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
        ax.grid(which='major', linestyle='-', linewidth='0.7', color='gray', alpha=0.3)
        ax.grid(which='minor', linestyle=':', linewidth='0.5', color='gray', alpha=0.15)
    axs[0, 0].plot(epochs, train_loss, label='Train Loss', color='orange', marker='.')
    axs[0, 0].plot(epochs, val_loss, label='Valid Loss', color='blue', marker='.')
    axs[0, 0].set_title("Loss Curves")
    axs[0, 0].set_ylabel("Loss")
    axs[0, 0].legend()
    make_smart_axis(axs[0, 0], y_step=0.1)
    axs[0, 1].plot(epochs, lrs, label='Learning Rate', color='red', linestyle='-')
    axs[0, 1].set_title("Learning Rate Schedule")
    axs[0, 1].set_ylabel("LR")
    axs[0, 1].set_yscale('log')
    axs[0, 1].legend()
    make_smart_axis(axs[0, 1], is_log=True)
    axs[1, 0].plot(epochs, val_iou_d, label='Disc IoU', color='purple', linestyle='-')
    axs[1, 0].plot(epochs, val_iou_c, label='Cup IoU', color='green', linestyle='-')
    axs[1, 0].set_title("Validation IoU")
    axs[1, 0].set_ylabel("IoU Score")
    axs[1, 0].legend()
    make_smart_axis(axs[1, 0], y_step=0.05)
    axs[1, 1].plot(epochs, val_dice_d, label='Disc Dice', color='purple', linestyle='-')
    axs[1, 1].plot(epochs, val_dice_c, label='Cup Dice', color='green', linestyle='-')
    axs[1, 1].set_title("Validation Dice Score")
    axs[1, 1].set_ylabel("Dice Score")
    axs[1, 1].legend()
    make_smart_axis(axs[1, 1], y_step=0.05)
    axs[2, 0].plot(epochs, val_prec_d, label='Disc Precision', color='purple', linestyle='-')
    axs[2, 0].plot(epochs, val_prec_c, label='Cup Precision', color='green', linestyle='-')
    axs[2, 0].set_title("Validation Precision")
    axs[2, 0].set_ylabel("Precision")
    axs[2, 0].legend()
    make_smart_axis(axs[2, 0], y_step=0.05)
    axs[2, 1].plot(epochs, val_rec_d, label='Disc Recall', color='purple', linestyle='-')
    axs[2, 1].plot(epochs, val_rec_c, label='Cup Recall', color='green', linestyle='-')
    axs[2, 1].set_title("Validation Recall (Sensitivity)")
    axs[2, 1].set_ylabel("Recall")
    axs[2, 1].legend()
    make_smart_axis(axs[2, 1], y_step=0.05)
    plt.show()
    if len(history) > 0:
        best_idx = np.argmax([(x.get('val_iou_disc',0) + x.get('val_iou_cup',0))/2 for x in history])
        best_ep = history[best_idx]
        print(f"BEST EPOCH: {best_ep['epoch']}")
        print(f"   IoU  (D/C): {best_ep.get('val_iou_disc',0):.4f} / {best_ep.get('val_iou_cup',0):.4f}")
        print(f"   Dice (D/C): {best_ep.get('val_dice_disc',0):.4f} / {best_ep.get('val_dice_cup',0):.4f}")
        print(f"   Prec (D/C): {best_ep.get('val_prec_disc',0):.4f} / {best_ep.get('val_prec_cup',0):.4f}")
        print(f"   Rec  (D/C): {best_ep.get('val_rec_disc',0):.4f} / {best_ep.get('val_rec_cup',0):.4f}")
plot_training_history()

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
import numpy as np
import os
import segmentation_models_pytorch as smp
SAVE_DIR = ''
BEST_MODEL_PATH = os.path.join(SAVE_DIR, "best_model.pth")
def calculate_final_metrics(pred_mask, true_mask):
    smooth = 1e-6
    p = pred_mask
    t = true_mask
    p_b = (p == 0).float()
    t_b = (t == 0).float()
    inter_b = (p_b * t_b).sum()
    union_b = p_b.sum() + t_b.sum() - inter_b
    dice_bg  = (2. * inter_b + smooth) / (p_b.sum() + t_b.sum() + smooth)
    iou_bg   = (inter_b + smooth) / (union_b + smooth)
    p_c = (p == 2).float()
    t_c = (t == 2).float()
    inter_c = (p_c * t_c).sum()
    union_c = p_c.sum() + t_c.sum() - inter_c
    dice_cup = (2. * inter_c + smooth) / (p_c.sum() + t_c.sum() + smooth)
    iou_cup  = (inter_c + smooth) / (union_c + smooth)
    p_d = ((p == 1) | (p == 2)).float()
    t_d = ((t == 1) | (t == 2)).float()
    inter_d = (p_d * t_d).sum()
    union_d = p_d.sum() + t_d.sum() - inter_d
    dice_disc = (2. * inter_d + smooth) / (p_d.sum() + t_d.sum() + smooth)
    iou_disc  = (inter_d + smooth) / (union_d + smooth)
    return iou_disc.item(), iou_cup.item(), iou_bg.item(), dice_disc.item(), dice_cup.item(), dice_bg.item()
print(f"Loading Best Model from: {BEST_MODEL_PATH}")
if not os.path.exists(BEST_MODEL_PATH):
    print("Error: Best model file not found! Did Cell 8 finish training?")
else:
    model = smp.UnetPlusPlus(
        encoder_name=Config.ENCODER, encoder_weights=None,
        in_channels=3, classes=3, activation=None
    ).to(Config.DEVICE)
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    print("Model Loaded Successfully")
    acc_iou_d, acc_iou_c, acc_iou_b = [], [], []
    acc_dice_d, acc_dice_c, acc_dice_b = [], [], []
    print("Starting Standard Evaluation on Test Set...")
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Testing"):
            images = images.to(Config.DEVICE)
            masks = masks.long().to(Config.DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            batch_size = images.size(0)
            for i in range(batch_size):
                id, ic, ib, dd, dc, db = calculate_final_metrics(preds[i], masks[i])
                acc_iou_d.append(id)
                acc_iou_c.append(ic)
                acc_iou_b.append(ib)
                acc_dice_d.append(dd)
                acc_dice_c.append(dc)
                acc_dice_b.append(db)
    print("\n" + "="*40)
    print("RESULTS")
    print("="*40)
    print(f"Background Dice:  {np.mean(acc_dice_b):.4f} (IoU: {np.mean(acc_iou_b):.4f})")
    print("-" * 20)
    print(f"Optic Disc Dice:  {np.mean(acc_dice_d):.4f}")
    print(f"Optic Cup Dice:   {np.mean(acc_dice_c):.4f}")
    print("-" * 20)
    print(f"Optic Disc IoU:   {np.mean(acc_iou_d):.4f}")
    print(f"Optic Cup IoU:    {np.mean(acc_iou_c):.4f}")
    print("="*40)